# Synthèse — certificats PEB wallons

Trois résultats SQL, trois graphiques matplotlib, des recommandations pour un acteur **public** et un acteur **privé**. Pas de dashboard : les figures sont générées depuis DuckDB (`sql/queries.sql`).

**Garde-fou.** On compare l’**Espec** (kWh/m².an), pas la part de G et pas Ew (absent de l’open data). Un certificat d’existant n’est pas un protocole de construction neuve. Détail : `docs/exploration.md`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "sql" / "queries.sql").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from synthese_figures import exporter_tout

# Régénère le diagramme ER + les 3 figures (idem : python scripts/synthese_figures.py).
for chemin in exporter_tout():
    print(chemin.relative_to(ROOT).as_posix())

## 1. Le neuf est vraiment plus sobre — surtout en Hainaut

Médiane Espec du neuf : **~83–91** kWh/m².an partout. L’existant va de **283** (Brabant wallon) à **362** (Hainaut). L’écart le plus large est en Hainaut (**+272**). Le neuf wallon est déjà homogène ; c’est le parc ancien hainuyer qui décroche.

Source : `sql/queries.sql` **Q03** (CTE + `RANK`).

![Écart Espec neuf vs existant par province](../pictures/readme/ecart-espec-provinces.png)

## 2. Pire taux ≠ plus gros volume à isoler

Hastière : **65 %** de F/G, mais peu de m². Charleroi : **~37 %** de F/G, **2,6 millions de m²** de passoires. Un classement « top 10 des pires communes » (Q02) et un classement « m² à rénover » (Q10) ne désignent pas les mêmes cibles.

![Taux de passoires vs volume en m²](../pictures/readme/taux-vs-volume-passoires.png)

## 3. Le levier technique est le chauffage du parc ancien

Poêles et électrique direct : Espec **400–480**, loin du neuf (88). Les **PAC** existantes (145) et la cogénération (102) se rapprochent déjà du neuf. Remplacer un poêle rapporte plus, en signal PEB, que de « chasser » le neuf (déjà A/B, 0 % de G).

Source : `sql/queries.sql` **Q07** (`RANK`). Les maisons existantes pèsent aussi plus que les appartements (Q05 : 389 vs 248).

![Espec selon le système de chauffage](../pictures/readme/chauffage-existant.png)

## Recommandations — acteur public (SPW, commune, SLSP)

1. **Deux listes, ou une seule.** Une enveloppe wallonne se calibre sur le **volume** (Charleroi, Liège, Mons, Namur). Un appel à projets « pires taux » peut traiter l’équité territoriale (Hastière, Honnelles, Rendeux) sans prétendre vider le gisement régional. La vue `v_priorite_renovation` fusionne les deux (60 % volume, 40 % taux) — Q11.
2. **Priorité Hainaut.** Pire existant *et* plus grand écart au neuf. Le Brabant wallon est déjà moins mal ; y mettre le même intensité dilue l’impact.
3. **Ne pas piloter au % de G neuf vs existant.** Le neuf n’a aucun G : l’indicateur est tautologique. L’Espec (et le volume de F/G en m²) est le thermomètre défendable.

## Recommandations — acteur privé (audit, ESCO, fournisseur)

1. **Cible commerciale mixte.** Volume en villes hainuyères et liégeoises ; niches rurales à très mauvais taux (audit, prime, offre « passoires »).
2. **Offre technique.** Maisons + poêles / électrique direct. Les PAC existantes ressemblent déjà au neuf : ce n’est pas là que le certificat bouge le plus.
3. **Ne pas vendre contre le neuf.** Le neuf wallon est déjà ~B ; la valeur est dans la rénovation de l’existant.

## Limites

- Les certificats ≠ le parc complet (biais vente / location / obligation).
- Protocoles existant et neuf distincts ; pont = Espec seulement.
- ~47 % des existants sans période de construction fiable (Q06).
- Grain = certificat ; l’extrait Parquet a des IDs uniques (pas de re-certif visible ici).

Schéma : `diagrams/modele_relationnel.png`. Requêtes : `sql/queries.sql`. Figures : `python scripts/synthese_figures.py`. Spatial : `sql/spatial.sql`.

## Annexe — stretch

**Priorité.** `v_priorite_renovation` combine Q02 et Q10 : `percent_rank` des m² F/G (60 %) et du taux F/G (40 %). Q11.

**Spatial.** Les certificats n’ont pas de XY. Les polygones SPW (Lambert 2008) donnent la densité au km² et les communes `ST_Touches` — pas de carte. `python scripts/load_spatial.py` puis `python scripts/run_spatial.py`.

In [ ]:
from pathlib import Path
import sys
import duckdb

ROOT = Path.cwd().resolve()
if not (ROOT / "sql" / "queries.sql").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

from run_queries import DB_PATH, query_by_id

con = duckdb.connect(str(DB_PATH), read_only=True)
print(con.sql(query_by_id("Q11")))
con.close()